## Impact of Program Design

A program that runs is not a program that works. One passing test says nothing about the input you didn't imagine -- and once code ships, it affects real people in ways its author never intended.

### Learning Targets

- I can explain system reliability and why one passing test isn't evidence of it.
- I can add validation that rejects invalid input before it corrupts an object.
- I can describe beneficial and harmful effects of the same program, including harm beyond its intended use.
- I can decide whether online code may be reused, based on its license.

## 1. Reference Guide

### Key Topics

<br>

| Term | Definition | Example |
|---|---|---|
| **System reliability** | A program behaves correctly across the full range of stated inputs, not just the one you tried. | One passing test with `targetTemp = 72` says nothing about `-400`. |
| **Guard clause** | A check at the top of a method that rejects invalid input before anything else runs. | `if (Double.isNaN(temp) \|\| temp < 50 \|\| temp > 90) throw ...` |
| **Silent failure** | Invalid input is accepted and produces a confident but wrong result, with no error. | `setTargetTemp(-400)` returning `HEATING` with no warning. |
| **Unintended consequence** | Harm beyond a program's intended use, even when the code runs exactly as designed. | Raising every thermostat 2 degrees helps the grid but harms one medically vulnerable resident. |
| **Open source license** | Legal terms that state what others may do with published code. | `MIT`, `Apache-2.0` |
| **`NaN`** | "Not a Number" -- a `double` value that fails every ordinary comparison (`<`, `>`, `==`). | `Double.NaN < 50` is `false`, so a range check alone won't catch it. |

### Guarding a Setter: Quick Checklist

- **Reject before assigning** -- the check runs first; the field only changes if the check passes.
- **Check `NaN` explicitly** -- it fails every `<`/`>` comparison, so it slips past a plain range check.
- **Fail loud** -- throw an exception with a clear message rather than silently ignoring bad input.
- **Leave the object unchanged on rejection** -- a failed call should never leave the object half-updated.

## 2. LxD Cycle Process

This lesson was built with the Learning Experience Design cycle -- every choice below traces to something observed about actual learners.

<div class="ocs__diagram">
<pre class="mermaid" style="margin:0;">flowchart LR
    E("<b>Empathize</b><br/>Students test one input,<br/>then move on") --> D("<b>Define</b><br/>A method owns its<br/>invalid input")
    D --> I("<b>Ideate</b><br/>Trace table with<br/>a wrong row")
    I --> P("<b>Prototype &amp; Test</b><br/>Worked example<br/>+ practice")</pre>
</div>

### Empathize

Watching classmates write their first validated methods, the same pattern showed up repeatedly: they tested one input, it returned the expected value, and they moved on. When asked "what happens if someone passes a negative number," the common answer was some version of *"why would they?"*

The misconception underneath it: **invalid input is something users do wrong, not something the method is responsible for handling.** Testing is treated as confirmation that the code works, not as an attempt to break it.

This is worth teaching because it produces no error. The code compiles, one test passes, and the failure surfaces later in a context where nobody is looking for it.

### Define

**Point of View:** A CSA student who can write a working method needs to see that a method owns its own invalid input -- they currently test to confirm success, not to find failure, which hides silent corruption.

### Ideate

**How Might We:** make a silent failure *visible* to someone whose code has never errored? Chosen activity: a trace table with a deliberately wrong row, so students watch a method accept -400 degrees and confidently report a status. `Thermostat` was chosen because an impossible value needs no domain knowledge to spot, and it scales into the impact sections below.

### Prototype & Test

The worked example, the broken-then-fixed pair, and the practice tasks below are that prototype -- taught and refined on the assigned teaching day.

## 3. College Board Requirements

This lesson's numbering (3.02) follows OCS's own Java catalog sequence, not a literal AP CSA Course and Exam Description unit/topic number -- the CED's official Unit 3 covers Boolean expressions and if-statements. The reliability, guard-clause, and impact content here draws on the CED's cross-cutting **Big Idea 5: Impact of Computing (IOC)**, which every AP CSA unit is expected to weave in alongside its primary topic (College Board, 2025).

- **Impact of Computing:** programs should be designed to behave predictably across the full range of possible inputs, and the people who write them are responsible for the beneficial and harmful effects of what they build, including effects beyond the intended use.
- **Licensing and reuse:** the CED expects students to understand that code found online carries intellectual property protections, and that reuse without a stated license or explicit permission is not safe to assume.

*Note: this section paraphrases CED guidance rather than quoting exact Learning Objective/Essential Knowledge codes, since this lesson doesn't map onto a single official CED sub-topic the way a unit like 1.2 does. If you have the specific IOC codes you'd like quoted verbatim, send them over and this section can be tightened to cite them directly.*

## 4. Lesson Plan

**Learning Objective:** Build a method that rejects invalid input before it changes an object's state, and explain a program's reliability, social impact, and licensing obligations in plain language.

**Success Criteria:** You can add a guard clause that runs before an assignment, trace what a guarded method does with a bad value, and decide whether a piece of found code is safe to reuse.

### Tech Talk (5 minutes)

A method that "works" on the input you tried can still be unreliable. **Reliability** means it behaves correctly across the *stated* range of inputs, not just the first one. A **guard clause** is the fix: check at the top of the method, and reject before you assign.

<pre><code class="language-java">
public void setTargetTemp(double temp) {
    if (Double.isNaN(temp) || temp &lt; 50 || temp &gt; 90) {
        throw new IllegalArgumentException("Target temp out of range: " + temp);
    }
    targetTemp = temp;   // only reached if the guard passed
}
</code></pre>

## The Running Example

**ClimateSense** is a hypothetical connected thermostat installed in tens of thousands of homes. Everything below centers on its `Thermostat` class, which holds a target temperature and reports the system's status:

<div class="ocs__diagram">
<pre class="mermaid" style="margin:0;">flowchart LR
    T(["targetTemp"]) --> A{"&ge; 78 ?"}
    A -- yes --> C(["COOLING"])
    A -- no --> B{"&le; 65 ?"}
    B -- yes --> H(["HEATING"])
    B -- no --> I(["IDLE"])</pre>
</div>

## Part 1: System Reliability

**System reliability** means a program performs as expected, under stated conditions, without failure -- not just for the one input you tried. Here is ClimateSense's unguarded first version. Predict all three outputs before reading on.

In [ ]:
// CODE_RUNNER: Predict all three outputs first, then press Run -- does anything look wrong?
public class Thermostat {
    private String roomName;
    private double targetTemp;

    public Thermostat(String roomName, double targetTemp) {
        this.roomName = roomName;
        this.targetTemp = targetTemp;
    }

    public void setTargetTemp(double temp) {
        targetTemp = temp;
    }

    public double getTargetTemp() { return targetTemp; }

    public String getStatus() {
        if (targetTemp >= 78) return "COOLING";
        if (targetTemp <= 65) return "HEATING";
        return "IDLE";
    }

    public static void main(String[] args) {
        Thermostat t = new Thermostat("Lab", 72);
        System.out.println(t.getStatus());

        t.setTargetTemp(80);
        System.out.println(t.getStatus());

        t.setTargetTemp(-400);          // below absolute zero
        System.out.println(t.getStatus());
    }
}
Thermostat.main(null);

| Call | `targetTemp` after | `getStatus()` | Correct? |
|---|---|---|---|
| `new Thermostat("Lab", 72)` | 72.0 | `IDLE` | Yes |
| `setTargetTemp(80)` | 80.0 | `COOLING` | Yes |
| `setTargetTemp(-400)` | -400.0 | `HEATING` | **No** -- below absolute zero |

No crash, no warning -- just a confident wrong answer. A silent failure is worse than a crash, because a crash tells you where to look.

### Popcorn Hacks

1. What does `getStatus()` return for exactly 78? For exactly 65?

In [ ]:
// CODE_RUNNER: Popcorn Hack 1 -- Press Run and read the status for exactly 78 and exactly 65.
public class Thermostat {
    private String roomName;
    private double targetTemp;

    public Thermostat(String roomName, double targetTemp) {
        this.roomName = roomName;
        this.targetTemp = targetTemp;
    }

    public void setTargetTemp(double temp) {
        targetTemp = temp;
    }

    public double getTargetTemp() { return targetTemp; }

    public String getStatus() {
        if (targetTemp >= 78) return "COOLING";
        if (targetTemp <= 65) return "HEATING";
        return "IDLE";
    }

    public static void main(String[] args) {
        Thermostat t = new Thermostat("Lab", 72);
        t.setTargetTemp(78);
        System.out.println("78 -> " + t.getStatus());
        t.setTargetTemp(65);
        System.out.println("65 -> " + t.getStatus());
    }
}
Thermostat.main(null);

## Part 2: Guarding Against Bad Input

A **guard clause** sits at the top of a method and rejects invalid input *before* anything else happens. The first version had none, so `-400`, `9999`, and `Double.NaN` all succeeded silently.

<div class="ocs__diagram">
<pre class="mermaid" style="margin:0;">flowchart LR
    S("setTargetTemp(temp)") --> V{"NaN, or outside<br/>50 - 90 ?"}
    V -- yes --> X("throw IllegalArgumentException<br/><b>object unchanged</b>")
    V -- no --> OK("targetTemp = temp<br/><b>value stored</b>")</pre>
</div>

In [ ]:
// CODE_RUNNER: Press Run to see the guard reject -400 and leave the object unchanged.
public class Thermostat {
    private String roomName;
    private double targetTemp;

    public Thermostat(String roomName, double targetTemp) {
        this.roomName = roomName;
        this.targetTemp = targetTemp;
    }

    public void setTargetTemp(double temp) {
        if (Double.isNaN(temp) || temp < 50 || temp > 90) {
            throw new IllegalArgumentException("Target temp out of range: " + temp);
        }
        targetTemp = temp;
    }

    public double getTargetTemp() { return targetTemp; }

    public String getStatus() {
        if (targetTemp >= 78) return "COOLING";
        if (targetTemp <= 65) return "HEATING";
        return "IDLE";
    }

    public static void main(String[] args) {
        Thermostat t = new Thermostat("Lab", 72);
        try {
            t.setTargetTemp(-400);
        } catch (IllegalArgumentException e) {
            System.out.println("rejected -> " + e.getMessage());
        }
        System.out.println(t.getTargetTemp());   // still 72.0 -- the exception ran before the assignment
    }
}
Thermostat.main(null);

A rejected call leaves the object exactly as it was -- failing is safer than corrupting. `NaN` needs its own check because it fails every comparison: `NaN < 50` and `NaN > 90` are both `false`, so a range check alone lets it through.

### Popcorn Hacks

1. Delete the `Double.isNaN(temp)` clause and predict what `setTargetTemp(Double.NaN)` does. Explain in one sentence.

In [ ]:
// CODE_RUNNER: Popcorn Hack 2 -- Press Run, then delete the Double.isNaN(temp) clause and run again. What changes?
public class Thermostat {
    private String roomName;
    private double targetTemp;

    public Thermostat(String roomName, double targetTemp) {
        this.roomName = roomName;
        this.targetTemp = targetTemp;
    }

    public void setTargetTemp(double temp) {
        if (Double.isNaN(temp) || temp < 50 || temp > 90) {
            throw new IllegalArgumentException("Target temp out of range: " + temp);
        }
        targetTemp = temp;
    }

    public double getTargetTemp() { return targetTemp; }

    public String getStatus() {
        if (targetTemp >= 78) return "COOLING";
        if (targetTemp <= 65) return "HEATING";
        return "IDLE";
    }

    public static void main(String[] args) {
        Thermostat t = new Thermostat("Lab", 72);
        try {
            t.setTargetTemp(Double.NaN);
            System.out.println("accepted -> " + t.getTargetTemp());
        } catch (IllegalArgumentException e) {
            System.out.println("rejected -> " + e.getMessage());
        }
    }
}
Thermostat.main(null);

## Part 3: Social, Economic, and Cultural Impact

One program can be beneficial and harmful at once -- the harm doesn't cancel the benefit. ClimateSense in fifty thousand homes:

<div class="ocs__diagram">
<pre class="mermaid" style="margin:0;">flowchart LR
    C("<b>ClimateSense</b><br/>50,000 homes")
    C --> E("Energy use")
    C --> S("Auto scheduling")
    C --> R("Remote access")
    E --> E1("Lower bills,<br/>less grid strain")
    S --> S1("Comfort without<br/>manual effort")
    S --> S2("Penalizes<br/>shift workers")
    R --> R1("Adjust before<br/>arriving")
    R --> R2("Needs smartphone<br/>+ home internet")</pre>
</div>

None of the harms are bugs -- they're consequences of reasonable design choices. Ask who's *excluded*, not just who's served.

## Part 4: Unintended Consequences

An **unintended consequence** is harm beyond a program's intended use, even when the code works exactly as designed. Suppose ClimateSense raises every target 2 degrees during a heat wave to ease grid load:

<div class="ocs__diagram">
<pre class="mermaid" style="margin:0;">flowchart LR
    A("<b>+2 degrees</b><br/>in one house") --> B("Barely noticeable")
    A --> C("x 50,000 houses at once<br/><b>demand shock on the grid</b>")
    A --> H("Medically vulnerable resident<br/><b>real health risk</b>")</pre>
</div>

Nobody wrote a harmful feature. The harm comes from **scale**, not intent.

## Part 5: Intellectual Property and Code Reuse

Reusing code is normal -- the rules are about *which* code, under *what terms*.

<div class="ocs__diagram">
<pre class="mermaid" style="margin:0;">flowchart TD
    Q("Found code online") --> L{"License stated?"}
    L -- no --> N("Treat as NOT free to use<br/><b>write your own</b>")
    L -- yes --> O{"Open source?"}
    O -- yes --> Y("Use it,<br/><b>follow the license</b>")
    O -- no --> P("<b>Only with permission</b>")</pre>
</div>

**Readable is not reusable.** A public repo is visible to everyone; that says nothing about permission.

### Popcorn Hacks

1. A classmate says a four-line snippet is too short for licensing to apply. Respond.

In [ ]:
// CODE_RUNNER: Popcorn Hack 3 -- A classmate says a four-line snippet is too short for licensing. Run it, then change license to "MIT" and "proprietary" and compare.
public class LicenseCheck {
    static String mayReuse(String license) {
        if (license.equals("none")) return "NO -- no license means no permission. Write your own.";
        if (license.equals("MIT") || license.equals("Apache-2.0")) return "YES -- follow the license (keep the notice).";
        if (license.equals("proprietary")) return "ONLY with the owner's permission.";
        return "Unknown license -- read it before reusing.";
    }

    public static void main(String[] args) {
        String snippet = "int add(int a, int b) { return a + b; }";   // short, but still someone's work
        String license = "none";                                      // try "MIT", "proprietary"

        System.out.println("Snippet: " + snippet);
        System.out.println("License: " + license);
        System.out.println("May I reuse it? " + mayReuse(license));
    }
}
LicenseCheck.main(null);

## Practice: Trace and Debug

### Predict the Output

Using the **guarded** `setTargetTemp`:

In [ ]:
// CODE_RUNNER: Pick your answer (A-D) below first, then press Run to check it.
public class Thermostat {
    private String roomName;
    private double targetTemp;

    public Thermostat(String roomName, double targetTemp) {
        this.roomName = roomName;
        this.targetTemp = targetTemp;
    }

    public void setTargetTemp(double temp) {
        if (Double.isNaN(temp) || temp < 50 || temp > 90) {
            throw new IllegalArgumentException("Target temp out of range: " + temp);
        }
        targetTemp = temp;
    }

    public double getTargetTemp() { return targetTemp; }

    public String getStatus() {
        if (targetTemp >= 78) return "COOLING";
        if (targetTemp <= 65) return "HEATING";
        return "IDLE";
    }

    public static void main(String[] args) {
        Thermostat t = new Thermostat("Lab", 72);
        try {
            t.setTargetTemp(95);
        } catch (IllegalArgumentException e) {
            System.out.println("rejected");
        }
        System.out.println(t.getTargetTemp());
        System.out.println(t.getStatus());
    }
}
Thermostat.main(null);

A. `95.0`, then `COOLING`

B. `rejected`, then `72.0`, then `IDLE`

C. `rejected`, then `95.0`, then `COOLING`

D. `rejected`, then `72.0`, then `COOLING`

### Popcorn Hacks: Apply the Idea

Finish `setRoomName` so it rejects `null`, empty, and whitespace-only names -- validate before assigning. The first attempt should be accepted and the next three rejected.

In [ ]:
// CODE_RUNNER: Popcorn Hack 4 -- Finish setRoomName so it rejects null, empty, and whitespace-only names, then press Run. Expect 1 accepted, 3 rejected.
public class Room {
    private String roomName;

    public Room(String roomName) { this.roomName = roomName; }

    public void setRoomName(String name) {
        // TODO: reject null, empty, and whitespace-only names.
        roomName = name;
    }

    public String getRoomName() { return roomName; }

    public static void main(String[] args) {
        Room r = new Room("Lab");
        String[] attempts = { "Nursery", null, "", "   " };
        for (String a : attempts) {
            try {
                r.setRoomName(a);
                System.out.println("accepted -> [" + r.getRoomName() + "]");
            } catch (IllegalArgumentException e) {
                System.out.println("rejected -> " + e.getMessage());
            }
        }
    }
}
Room.main(null);

## Answer Check

**Predict the Output -- B.** 95 is outside 50-90, so the guard throws before the assignment; the object keeps 72.0 and `getStatus()` returns `IDLE`. (A and C assume the value was stored anyway; D contradicts its own state.)

**Apply the Idea.** Reject `null` (a later method call on it would throw `NullPointerException` far from the real cause) and empty or whitespace-only names (an unidentifiable room is unusable). The rule: a mutator rejects anything that leaves the object unable to function.

## Quick Review

- **Reliability:** works under all stated conditions, not just the one input you tried.
- **Test edges:** zero, negative, empty, `null`, `NaN`, boundaries.
- **Guard clause:** runs before assignment, so invalid input is never stored.
- **Silent acceptance** is more dangerous than a crash.
- **Impact:** one program can help and harm, and harm beyond intended use comes from scale, not intent.
- **Reuse:** visible code isn't free code. No license means no permission.

## 6. Hacks & Practice Tasks

### Prepare your submission IPYNB

1. Create a new notebook in your portfolio homework area: `_notebooks/homework`.
2. Add one raw cell at the top with the frontmatter:

<pre><code>---
layout: post
codemirror: true
title: Impact of Program Design HW
categories: [Java]
lesson_language: Java
lesson_topic: Impact-of-Program-Design HW
lesson_part: interactive
lesson_type: lesson
permalink: /csa/unit_03/3_2-hw
author: yourGithubID
---
</code></pre>

3. Add code cells for the four Popcorn Hacks above and the Homework Hack below. Make sure every cell runs with visible output.
4. Submit the link to your published page at the bottom of this page, and paste this in the description box:

<pre><code>Lesson: CSA 3.2 Impact of Program Design
MCQ 3.2: &lt;paste the copied result line, such as 4/4 | answers: B,B,B,C&gt;
Popcorn 1-4: all four hacks run with visible output (yes/no)
Homework: setHumidity rejects NaN and out-of-range values (yes/no)
Homework: accepted/rejected output printed for all four test values (yes/no)
</code></pre>

### Submission Safety Rules (Read First)

- One class per cell, ending with `ClassName.main(null);`.
- Run each cell and leave the output showing.
- Use your own test values where the hack asks for them, not only the sample answer.
- Include your MCQ score.
- Use `##`/`###` headings or smaller.

### MCQ Check

4 questions, one at a time. Answer, check, then go to the next one. At the end, copy the score line into your submission notes.

<section class="ocs__dnd" id="mcq-csa-3-2"></section>
<script>
(function () {
  var data = [{"q": "ClimateSense's first version passed one test with targetTemp = 72, returning IDLE. What does that one passing test actually prove?", "choices": ["The method works correctly for every input", "The method works for that one input; it says nothing about other inputs", "The method has no bugs", "The method rejects invalid input"], "answer": 1, "ref": "System Reliability", "why": "A single passing test only confirms behavior for the input tried -- reliability requires correctness under all stated conditions."}, {"q": "Why does the guarded setTargetTemp need a separate Double.isNaN(temp) check instead of relying on temp < 50 || temp > 90 alone?", "choices": ["NaN is faster to compare than normal numbers", "NaN < 50 and NaN > 90 are both false, so a range check alone lets NaN through", "Java doesn't allow NaN in double variables", "isNaN() throws an exception automatically"], "answer": 1, "ref": "Guard Clauses", "why": "NaN fails every ordinary comparison, so a range check alone never catches it -- it needs its own explicit check."}, {"q": "ClimateSense's auto-scheduling feature saves energy for most households but makes things worse for night-shift workers whose schedules don't match the default pattern. What does this illustrate?", "choices": ["A bug in the scheduling algorithm", "The same feature can be beneficial and harmful at once, without being a bug", "Night-shift workers misused the feature", "The feature should be deleted"], "answer": 1, "ref": "Social, Economic, and Cultural Impact", "why": "One program can help most users and still create real costs for others -- the harm doesn't cancel the benefit, and it isn't a bug."}, {"q": "You find a four-line Java snippet in a public GitHub repo with no LICENSE file. Can you use it in your project?", "choices": ["Yes, anything posted publicly is free to use", "Yes, because it's too short to be covered by copyright", "No -- no license means no permission; treat it as not free to use", "Yes, as long as you don't say you wrote it"], "answer": 2, "ref": "Intellectual Property and Code Reuse", "why": "Visibility is not permission. With no stated license, the safest and legally correct assumption is that the code is not free to reuse."}];
  var root = document.getElementById("mcq-csa-3-2");
  if (!root) return;
  var index = 0;
  var correct = 0;
  var picks = [];

  function clear() { while (root.firstChild) { root.removeChild(root.firstChild); } }

  function showQuestion() {
    clear();
    var item = data[index];
    var progress = document.createElement("p");
    progress.className = "ocs__dnd-progress";
    progress.textContent = "Question " + (index + 1) + " of " + data.length;
    root.appendChild(progress);
    var question = document.createElement("p");
    question.className = "ocs__dnd-status";
    question.textContent = item.q;
    root.appendChild(question);
    if (item.code) {
      var pre = document.createElement("pre");
      var code = document.createElement("code");
      code.textContent = item.code;
      pre.appendChild(code);
      root.appendChild(pre);
    }
    var form = document.createElement("form");
    var panel = document.createElement("div");
    panel.className = "ocs__dnd-panel";
    var picked = null;
    var choiceButtons = item.choices.map(function (choice, j) {
      var choiceButton = document.createElement("button");
      choiceButton.type = "button";
      choiceButton.className = "ocs__drop-btn";
      choiceButton.setAttribute("aria-pressed", "false");
      choiceButton.textContent = "ABCD".charAt(j) + ". " + choice;
      choiceButton.addEventListener("click", function () {
        picked = j;
        choiceButtons.forEach(function (other, k) {
          other.classList.toggle("is-over", k === j);
          other.setAttribute("aria-pressed", String(k === j));
        });
      });
      panel.appendChild(choiceButton);
      return choiceButton;
    });
    form.appendChild(panel);
    var feedback = document.createElement("p");
    feedback.className = "ocs__dnd-progress";
    feedback.setAttribute("aria-live", "polite");
    var buttons = document.createElement("div");
    var check = document.createElement("button");
    check.type = "submit";
    check.className = "ocs__dnd-reset";
    check.textContent = "Check answer";
    buttons.appendChild(check);
    form.appendChild(feedback);
    form.appendChild(buttons);
    form.addEventListener("submit", function (event) {
      event.preventDefault();
      if (picked === null) { feedback.textContent = "Pick an answer first."; return; }
      var value = picked;
      picks.push("ABCD".charAt(value));
      if (value === item.answer) {
        correct++;
        feedback.textContent = "Correct. " + item.why;
      } else {
        feedback.textContent = "Not quite. The answer is " + "ABCD".charAt(item.answer) + ". " + item.why + " (Review " + item.ref + ".)";
      }
      choiceButtons.forEach(function (choiceButton) { choiceButton.disabled = true; });
      choiceButtons[item.answer].classList.add("is-filled");
      buttons.removeChild(check);
      var next = document.createElement("button");
      next.type = "button";
      next.className = "ocs__dnd-reset";
      next.textContent = index + 1 < data.length ? "Next question" : "See my score";
      next.addEventListener("click", function () {
        index++;
        if (index < data.length) { showQuestion(); } else { showScore(); }
      });
      buttons.appendChild(next);
    });
    root.appendChild(form);
  }

  function showScore() {
    clear();
    var score = document.createElement("p");
    score.className = "ocs__dnd-status";
    score.textContent = "Score: " + correct + " / " + data.length;
    root.appendChild(score);
    var line = document.createElement("div");
    line.className = "ocs__dnd-panel";
    var hint = document.createElement("label");
    hint.textContent = "Paste this line into your submission notes:";
    var box = document.createElement("input");
    box.type = "text";
    box.className = "ocs__dnd-input";
    box.readOnly = true;
    box.value = "MCQ 3.2: " + correct + "/" + data.length + " | answers: " + picks.join(",");
    var copy = document.createElement("button");
    copy.type = "button";
    copy.className = "ocs__dnd-reset";
    copy.textContent = "Copy result";
    copy.addEventListener("click", function () {
      box.select();
      if (navigator.clipboard) { navigator.clipboard.writeText(box.value); }
      copy.textContent = "Copied";
      setTimeout(function () { copy.textContent = "Copy result"; }, 1500);
    });
    var again = document.createElement("button");
    again.type = "button";
    again.className = "ocs__dnd-reset";
    again.textContent = "Try again";
    again.addEventListener("click", function () { index = 0; correct = 0; picks = []; showQuestion(); });
    line.appendChild(hint);
    line.appendChild(box);
    var actions = document.createElement("p");
    actions.appendChild(copy);
    actions.appendChild(document.createTextNode(" "));
    actions.appendChild(again);
    line.appendChild(actions);
    root.appendChild(line);
  }

  showQuestion();
})();
</script>

### Homework Hack

**Task:** `HumidityStation` has no guard clause yet. Finish `setHumidity` so it rejects `NaN` and any value outside `0-100` *before* assigning, then run the four test values and confirm one is accepted and three are rejected.

**Solution Skeleton:**

In [ ]:
// CODE_RUNNER: Homework Hack -- finish setHumidity so it rejects NaN and values outside 0-100, then press Run. Expect 1 accepted, 3 rejected.
public class HumidityStation {
    private double humidity;

    public HumidityStation(double humidity) { this.humidity = humidity; }

    public void setHumidity(double value) {
        // TODO: reject NaN and any value outside 0-100 before assigning.
        humidity = value;
    }

    public double getHumidity() { return humidity; }

    public static void main(String[] args) {
        HumidityStation station = new HumidityStation(45);
        double[] attempts = { 55, -10, 150, Double.NaN };
        for (double a : attempts) {
            try {
                station.setHumidity(a);
                System.out.println("accepted -> " + station.getHumidity());
            } catch (IllegalArgumentException e) {
                System.out.println("rejected -> " + e.getMessage());
            }
        }
    }
}
HumidityStation.main(null);

### Grading Plan (1 Point Total)

| Part | Points | What earns the points |
| --- | --- | --- |
| Popcorn Hacks 1-4 | 0.3 | All four hacks completed, and every cell runs with visible output. |
| MCQ | 0.2 | 3 or 4 correct. 0.1 if every question was answered. |
| Homework: guard clause | 0.25 | `setHumidity` rejects `NaN` and out-of-range values before assigning. |
| Homework: trace | 0.25 | Correct accepted/rejected output printed for all four test values. |
| **Total** | **1.0** | |

#### Quick Validation Checklist

- Each cell ends with `ClassName.main(null);` and shows output.
- MCQ score in the notes.
- `setHumidity` checks `NaN` and the `0-100` range before assigning.
- All four test values print the expected accepted/rejected result.

## 7. Lesson Revisions

**Revision Made:**

- Restructured the lesson into the same numbered-section format as other Java-catalog lessons (e.g. 1.2 Variables and Data Types): added `## 1. Reference Guide`, `## 3. College Board Requirements`, and `## 4. Lesson Plan`, and renumbered the existing LxD cycle section to `## 2. LxD Cycle Process`.
- Added the missing assessment and submission scaffolding under `## 6. Hacks & Practice Tasks`: a scored **MCQ Check**, a standalone **Homework Hack** separate from the Popcorn Hacks, a weighted **Grading Plan**, and the "Prepare your submission IPYNB" instructions -- plus this `## 7. Lesson Revisions` and the `## 8. Feedback Evidence` section.
- Replaced all 6 mermaid diagrams' hardcoded `%%{init: ...}%%` theme and `classDef` color overrides with the site's shared `<div class="ocs__diagram"><pre class="mermaid">` convention, so diagrams follow the global SASS theme instead of fixed hex values -- also cutting about 90 lines of repeated boilerplate.

## 8. Feedback Evidence

**Feedback Received:** Darshan S. reviewed the lesson and noted it runs very long -- even though the content itself is appropriately in-depth, the overall length is a concern and worth revisiting in a future pass (e.g. trimming prose, splitting into a shorter core lesson plus optional extension material, or cutting a Part section).

## 9. References

College Board. (2025). *AP Computer Science A course and exam description*, Unit 3, Topic 3.02 -- Impact of Program Design [Effective fall 2025]. https://apcentral.collegeboard.org/media/pdf/ap-computer-science-a-course-and-exam-description-effective-fall-2025.pdf

Oracle. (n.d.). *Double (Java SE 17 & JDK 17)*. https://docs.oracle.com/en/java/javase/17/docs/api/java.base/java/lang/Double.html -- `isNaN` and `NaN` comparison behavior used by `Thermostat.setTargetTemp`.

Oracle. (n.d.). *IllegalArgumentException (Java SE 17 & JDK 17)*. https://docs.oracle.com/en/java/javase/17/docs/api/java.base/java/lang/IllegalArgumentException.html -- the exception thrown by the guard clause in this lesson's `Thermostat` and `Room` examples.

Open Source Initiative. (n.d.). *The Open Source Definition*. https://opensource.org/osd -- referenced in Part 5's license-check example (`LicenseCheck.mayReuse`).

GitHub Docs. (n.d.). *Licensing a repository*. https://docs.github.com/en/repositories/managing-your-repositorys-settings-and-features/customizing-your-repository/licensing-a-repository -- referenced in Part 5's discussion of reusing code found online.